# MedScribe Agent - Setup & Verification

This notebook verifies the development environment and runs the end-to-end pipeline in **demo mode** (no GPU required).

For real MedGemma 4B inference on Kaggle T4, see `02_kaggle_medgemma.ipynb`.

## Prerequisites

```bash
# From the HealthChain repo root:
cd medscribe
uv pip install -e /path/to/HealthChain  # Install HealthChain from local source
uv sync                                  # Install medscribe dependencies

# Install scispaCy model:
uv pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

# For real MedGemma inference (requires CUDA GPU):
# 1. Accept license at https://huggingface.co/google/medgemma-4b-it
# 2. huggingface-cli login
# 3. export MEDGEMMA_MODE=real
```

## 1. Environment Verification

In [ ]:
import sys
print(f"Python: {sys.version}")

# Core dependencies
import healthchain
print(f"HealthChain: {healthchain.__version__}")

import spacy
print(f"spaCy: {spacy.__version__}")

import scispacy
print(f"scispaCy: {scispacy.__version__}")

import yaml
print(f"PyYAML: {yaml.__version__}")

# HuggingFace inference stack
import transformers
print(f"transformers: {transformers.__version__}")

try:
    import bitsandbytes
    print(f"bitsandbytes: {bitsandbytes.__version__}")
except ImportError:
    print("bitsandbytes: NOT INSTALLED (needed for 4-bit quantization on GPU)")

try:
    import accelerate
    print(f"accelerate: {accelerate.__version__}")
except ImportError:
    print("accelerate: NOT INSTALLED (needed for device_map='auto')")

# GPU check
import torch
print(f"\ntorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check if scispaCy model is installed
try:
    nlp = spacy.load("en_core_sci_sm")
    print(f"\nen_core_sci_sm: loaded ({len(nlp.pipe_names)} pipes)")
except OSError:
    print("\nWARNING: en_core_sci_sm not installed. Run:")
    print("  pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz")

print("\nEnvironment OK")

## 2. Test scispaCy NLP Pipeline

In [ ]:
import sys
import os

# Add project root to path for imports
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from healthchain.io import Document

SAMPLE_NOTE = """Patient is a 65-year-old male admitted for community-acquired pneumonia.
Past medical history significant for type 2 diabetes mellitus, hypertension,
and hyperlipidemia. Currently on metformin 1000mg BID, lisinopril 20mg daily,
and atorvastatin 40mg at bedtime. Chest X-ray shows right lower lobe infiltrate.
Started on azithromycin 500mg IV and ceftriaxone 1g IV.
Labs show WBC 14.2, glucose 185 mg/dL, HbA1c 7.8%.
Assessment: Community-acquired pneumonia, uncontrolled type 2 diabetes."""

print(f"Sample note: {len(SAMPLE_NOTE)} chars")
print(SAMPLE_NOTE[:200] + "...")

In [ ]:
# Try full pipeline with linker, fall back to simple
try:
    from medscribe.src.pipeline.coding_pipeline import build_coding_pipeline
    pipeline = build_coding_pipeline()
    print("Loaded full pipeline with UMLS entity linker")
except Exception as e:
    print(f"Could not load full pipeline: {e}")
    print("Falling back to simple pipeline (NER only)...")
    from medscribe.src.pipeline.coding_pipeline import build_coding_pipeline_simple
    pipeline = build_coding_pipeline_simple()
    print("Loaded simple pipeline (NER only, no CUI linking)")

doc = Document(SAMPLE_NOTE)
result = pipeline(doc)

# Show NLP entities
entities = result.nlp.get_entities()
print(f"\nExtracted {len(entities)} entities:")
for e in entities:
    print(f"  {e.get('text', 'N/A'):30s} [{e.get('label', 'N/A')}]")

# Show spaCy entities with CUI if available
spacy_doc = result.nlp.get_spacy_doc()
if spacy_doc:
    print(f"\nspaCy entities ({len(spacy_doc.ents)}):\n")
    for ent in spacy_doc.ents:
        cui = getattr(ent._, 'cui', None) if hasattr(ent._, 'cui') else None
        cui_str = f" -> CUI: {cui}" if cui else ""
        print(f"  {ent.text:30s} [{ent.label_}]{cui_str}")

# Show FHIR problem list
problems = result.fhir.problem_list
print(f"\nFHIR Problem List ({len(problems)} conditions):")
for cond in problems:
    if cond.code and cond.code.coding:
        c = cond.code.coding[0]
        print(f"  {c.display} ({c.code})")

## 3. Test Demo MedGemma Client

In [ ]:
import json
from medscribe.src.models.medgemma import create_client

# Force demo mode (no GPU needed for this notebook)
# For real inference, use 02_kaggle_medgemma.ipynb or set MEDGEMMA_MODE=real
client = create_client(mode="demo")
print(f"Client type: {type(client).__name__}")

# Run reasoning
result = client.reason_over_note(SAMPLE_NOTE, entities)
print(f"\nMedGemma Response:")
print(json.dumps(result, indent=2))

In [ ]:
# Test resolution endpoint
discrepancies = [
    {"type": "nlp_only", "entity": "WBC 14.2", "reason": "Found by NLP but not confirmed by LLM"}
]
resolution = client.reason_with_resolution(SAMPLE_NOTE, discrepancies)
print("Resolution Response:")
print(json.dumps(resolution, indent=2))

## 4. Full End-to-End Agent Run

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from medscribe.src.agent.orchestrator import MedScribeAgent

# Build agent with demo client, no FHIR gateway (standalone mode)
agent = MedScribeAgent(
    coding_pipeline=pipeline,
    medgemma_client=client,
    fhir_gateway=None,  # No live FHIR server for Phase 1
)

# Run the full pipeline
output = agent.run(SAMPLE_NOTE, patient_id="Patient/demo-001")

print(f"\n{'='*60}")
print("AGENT RUN COMPLETE")
print(f"{'='*60}")

## 5. Inspect Output

In [ ]:
# NLP Entities
print(f"NLP Entities: {len(output['nlp_entities'])}")
for e in output['nlp_entities'][:10]:
    print(f"  {e.get('text', 'N/A'):30s} [{e.get('label', 'N/A')}]")

print(f"\nLLM Diagnoses: {len(output['llm_reasoning'].get('diagnoses', []))}")
for dx in output['llm_reasoning'].get('diagnoses', []):
    print(f"  {dx['text']:30s} {dx['icd10']:10s} [{dx['confidence']}] {dx.get('gaps', '')}")

print(f"\nDiscrepancies: {len(output['discrepancies'])}")
for d in output['discrepancies']:
    print(f"  [{d['type']}] {d.get('entity', 'N/A')}: {d.get('reason', '')}")

In [ ]:
# CDS Cards
print(f"CDS Cards ({len(output['cds_cards'])}):\n")
for card in output['cds_cards']:
    indicator = card['indicator'].upper()
    print(f"  [{indicator:8s}] {card['summary']}")
    print(f"             {card['detail']}")
    print()

In [ ]:
# FHIR Bundle
bundle = output['fhir_bundle']
if bundle:
    print(f"FHIR Bundle type: {bundle.type}")
    entries = bundle.entry or []
    print(f"Entries: {len(entries)}")
    for entry in entries:
        r = entry.resource
        print(f"  {r.resource_type}: {r.code.text if hasattr(r, 'code') and r.code else 'N/A'}")
else:
    print("No FHIR Bundle generated (expected when using simple pipeline without UMLS linker)")